# 1. Analysis - Balance Sheet

## Notebook Summary

This notebook analyzes a single company’s balance sheet history from Financial Modeling Prep using the shared ticker in `../../params.py`. It is organized into data loading and preparation, annual and quarterly summary snapshots, long-horizon balance sheet trend charts, and common-size, liquidity, and capital-structure diagnostics.

Run the notebook from top to bottom after updating `ticker_str` in `../../params.py` so the helper functions, balance sheet datasets, and all downstream tables and charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly balance sheet history, and calculate the derived liquidity, leverage, and common-size fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Setup and FMP helpers
from pathlib import Path
import os
import runpy

import pandas as pd
import plotly.graph_objects as go
import requests


PARAMS_FILE_CANDIDATES = []
for base_path in [Path.cwd(), *Path.cwd().parents]:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )

for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]
BASE_URL = "https://financialmodelingprep.com/stable"
ANNUAL_LIMIT = 20
QUARTERLY_LIMIT = 40


def find_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return current


def load_project_env() -> None:
    env_path = find_project_root() / ".env"
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def prepare_balance_sheet_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if frame.empty:
        return frame

    frame["date"] = pd.to_datetime(frame.get("date"), errors="coerce")

    if "calendarYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["calendarYear"], errors="coerce")
    elif "fiscalYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["fiscalYear"], errors="coerce")
    else:
        frame["calendarYear"] = pd.NA

    if frame["calendarYear"].isna().all():
        frame["calendarYear"] = frame["date"].dt.year

    if "period" not in frame.columns:
        frame["period"] = pd.NA

    numeric_columns = [
        "cashAndCashEquivalents",
        "shortTermInvestments",
        "cashAndShortTermInvestments",
        "netReceivables",
        "inventory",
        "otherCurrentAssets",
        "totalCurrentAssets",
        "propertyPlantEquipmentNet",
        "goodwill",
        "intangibleAssets",
        "goodwillAndIntangibleAssets",
        "longTermInvestments",
        "otherNonCurrentAssets",
        "totalNonCurrentAssets",
        "totalAssets",
        "accountPayables",
        "shortTermDebt",
        "totalCurrentLiabilities",
        "longTermDebt",
        "totalDebt",
        "otherNonCurrentLiabilities",
        "totalNonCurrentLiabilities",
        "totalLiabilities",
        "retainedEarnings",
        "totalStockholdersEquity",
        "totalEquity",
        "netDebt",
    ]
    for column in numeric_columns:
        if column not in frame.columns:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame


def apply_standard_figure_layout(fig: go.Figure, title: str, height: int, bottom_margin: int = 140) -> None:
    fig.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_dark",
        paper_bgcolor="#020817",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="x unified",
        hoverlabel={
            "bgcolor": "#0f172a",
            "font": {"color": "#e2e8f0", "size": 13},
            "namelength": -1,
        },
        legend={
            "orientation": "h",
            "yanchor": "top",
            "y": -0.16,
            "x": 0,
            "xanchor": "left",
            "bgcolor": "rgba(2, 8, 23, 0.6)",
        },
        autosize=True,
        height=height,
        margin={"l": 60, "r": 30, "t": 120, "b": bottom_margin},
    )


def format_metric_value(value: float, style: str) -> str:
    if pd.isna(value):
        return "-"
    if style == "currency":
        return f"${value / 1_000_000_000:,.2f}B"
    if style == "percent":
        return f"{value:.1%}"
    if style == "ratio":
        return f"{value:,.2f}"
    return f"{value:,.2f}"


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
# 3. Load annual balance sheet history
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

annual_balance_sheet = prepare_balance_sheet_frame(
    pd.DataFrame(request_json("balance-sheet-statement", symbol=SYMBOL, limit=ANNUAL_LIMIT))
)

if annual_balance_sheet.empty:
    raise RuntimeError(f"FMP did not return annual balance sheet history for {SYMBOL}.")

annual_balance = (
    annual_balance_sheet.loc[:, [
        "date" ,
        "calendarYear" ,
        "period" ,
        "cashAndCashEquivalents" ,
        "shortTermInvestments" ,
        "cashAndShortTermInvestments" ,
        "netReceivables" ,
        "inventory" ,
        "otherCurrentAssets" ,
        "totalCurrentAssets" ,
        "propertyPlantEquipmentNet" ,
        "goodwill" ,
        "intangibleAssets" ,
        "goodwillAndIntangibleAssets" ,
        "longTermInvestments" ,
        "otherNonCurrentAssets" ,
        "totalNonCurrentAssets" ,
        "totalAssets" ,
        "accountPayables" ,
        "shortTermDebt" ,
        "totalCurrentLiabilities" ,
        "longTermDebt" ,
        "totalDebt" ,
        "otherNonCurrentLiabilities" ,
        "totalNonCurrentLiabilities" ,
        "totalLiabilities" ,
        "retainedEarnings" ,
        "totalStockholdersEquity" ,
        "totalEquity" ,
        "netDebt" ,
    ]]
    .dropna(subset=["date", "totalAssets"])
)
annual_balance["totalEquity"] = annual_balance["totalEquity"].fillna(annual_balance["totalStockholdersEquity"])
annual_balance["totalStockholdersEquity"] = annual_balance["totalStockholdersEquity"].fillna(annual_balance["totalEquity"])
annual_balance["cashAndShortTermInvestments"] = annual_balance["cashAndShortTermInvestments"].fillna(
    annual_balance["cashAndCashEquivalents"].fillna(0) + annual_balance["shortTermInvestments"].fillna(0)
)
annual_balance["goodwillAndIntangibleAssets"] = annual_balance["goodwillAndIntangibleAssets"].fillna(
    annual_balance["goodwill"].fillna(0) + annual_balance["intangibleAssets"].fillna(0)
)
annual_balance["totalDebt"] = annual_balance["totalDebt"].fillna(
    annual_balance["shortTermDebt"].fillna(0) + annual_balance["longTermDebt"].fillna(0)
)
annual_balance["totalNonCurrentAssets"] = annual_balance["totalNonCurrentAssets"].fillna(
    annual_balance["totalAssets"] - annual_balance["totalCurrentAssets"]
)
annual_balance["totalNonCurrentLiabilities"] = annual_balance["totalNonCurrentLiabilities"].fillna(
    annual_balance["totalLiabilities"] - annual_balance["totalCurrentLiabilities"]
)
annual_balance["netDebt"] = annual_balance["netDebt"].fillna(
    annual_balance["totalDebt"] - annual_balance["cashAndCashEquivalents"].fillna(0)
)
annual_balance = annual_balance.sort_values("date").reset_index(drop=True)
annual_balance["workingCapital"] = annual_balance["totalCurrentAssets"] - annual_balance["totalCurrentLiabilities"]
annual_balance["cashAndShortTermInvestmentsPctAssets"] = annual_balance["cashAndShortTermInvestments"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["receivablesPctAssets"] = annual_balance["netReceivables"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["inventoryPctAssets"] = annual_balance["inventory"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["ppePctAssets"] = annual_balance["propertyPlantEquipmentNet"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["goodwillIntangiblePctAssets"] = annual_balance["goodwillAndIntangibleAssets"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["longTermInvestmentsPctAssets"] = annual_balance["longTermInvestments"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["currentAssetsPctAssets"] = annual_balance["totalCurrentAssets"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["nonCurrentAssetsPctAssets"] = annual_balance["totalNonCurrentAssets"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["currentLiabilitiesPctAssets"] = annual_balance["totalCurrentLiabilities"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["nonCurrentLiabilitiesPctAssets"] = annual_balance["totalNonCurrentLiabilities"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["totalDebtPctAssets"] = annual_balance["totalDebt"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["totalLiabilitiesPctAssets"] = annual_balance["totalLiabilities"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["totalEquityPctAssets"] = annual_balance["totalEquity"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["currentRatio"] = annual_balance["totalCurrentAssets"].div(annual_balance["totalCurrentLiabilities"].replace(0, pd.NA))
annual_balance["cashRatio"] = annual_balance["cashAndShortTermInvestments"].div(annual_balance["totalCurrentLiabilities"].replace(0, pd.NA))
annual_balance["workingCapitalPctAssets"] = annual_balance["workingCapital"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["debtToAssets"] = annual_balance["totalDebt"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["liabilityToAssets"] = annual_balance["totalLiabilities"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["equityRatio"] = annual_balance["totalEquity"].div(annual_balance["totalAssets"].replace(0, pd.NA))
annual_balance["debtToEquity"] = annual_balance["totalDebt"].div(annual_balance["totalEquity"].replace(0, pd.NA))
annual_balance["netDebtToEquity"] = annual_balance["netDebt"].div(annual_balance["totalEquity"].replace(0, pd.NA))
annual_balance["totalAssetsYoY"] = annual_balance["totalAssets"].pct_change()
annual_balance["totalLiabilitiesYoY"] = annual_balance["totalLiabilities"].pct_change()
annual_balance["totalEquityYoY"] = annual_balance["totalEquity"].pct_change()
annual_balance["cashYoY"] = annual_balance["cashAndShortTermInvestments"].pct_change()
annual_balance["totalDebtYoY"] = annual_balance["totalDebt"].pct_change()
annual_balance["workingCapitalYoY"] = annual_balance["workingCapital"].pct_change()
annual_balance["currentRatioChange"] = annual_balance["currentRatio"].diff()
annual_balance["debtToEquityChange"] = annual_balance["debtToEquity"].diff()
annual_balance["liabilityToAssetsChange"] = annual_balance["liabilityToAssets"].diff()
annual_balance["equityRatioChange"] = annual_balance["equityRatio"].diff()

In [ ]:
# 4. Load quarterly balance sheet history
quarterly_balance_sheet = prepare_balance_sheet_frame(
    pd.DataFrame(request_json("balance-sheet-statement", symbol=SYMBOL, period="quarter", limit=QUARTERLY_LIMIT))
)

if quarterly_balance_sheet.empty:
    raise RuntimeError(f"FMP did not return quarterly balance sheet history for {SYMBOL}.")

quarterly_balance = (
    quarterly_balance_sheet.loc[:, [
        "date" ,
        "calendarYear" ,
        "period" ,
        "cashAndCashEquivalents" ,
        "shortTermInvestments" ,
        "cashAndShortTermInvestments" ,
        "netReceivables" ,
        "inventory" ,
        "otherCurrentAssets" ,
        "totalCurrentAssets" ,
        "propertyPlantEquipmentNet" ,
        "goodwill" ,
        "intangibleAssets" ,
        "goodwillAndIntangibleAssets" ,
        "longTermInvestments" ,
        "otherNonCurrentAssets" ,
        "totalNonCurrentAssets" ,
        "totalAssets" ,
        "accountPayables" ,
        "shortTermDebt" ,
        "totalCurrentLiabilities" ,
        "longTermDebt" ,
        "totalDebt" ,
        "otherNonCurrentLiabilities" ,
        "totalNonCurrentLiabilities" ,
        "totalLiabilities" ,
        "retainedEarnings" ,
        "totalStockholdersEquity" ,
        "totalEquity" ,
        "netDebt" ,
    ]]
    .dropna(subset=["date", "totalAssets"])
)
quarterly_balance["totalEquity"] = quarterly_balance["totalEquity"].fillna(quarterly_balance["totalStockholdersEquity"])
quarterly_balance["totalStockholdersEquity"] = quarterly_balance["totalStockholdersEquity"].fillna(quarterly_balance["totalEquity"])
quarterly_balance["cashAndShortTermInvestments"] = quarterly_balance["cashAndShortTermInvestments"].fillna(
    quarterly_balance["cashAndCashEquivalents"].fillna(0) + quarterly_balance["shortTermInvestments"].fillna(0)
)
quarterly_balance["goodwillAndIntangibleAssets"] = quarterly_balance["goodwillAndIntangibleAssets"].fillna(
    quarterly_balance["goodwill"].fillna(0) + quarterly_balance["intangibleAssets"].fillna(0)
)
quarterly_balance["totalDebt"] = quarterly_balance["totalDebt"].fillna(
    quarterly_balance["shortTermDebt"].fillna(0) + quarterly_balance["longTermDebt"].fillna(0)
)
quarterly_balance["totalNonCurrentAssets"] = quarterly_balance["totalNonCurrentAssets"].fillna(
    quarterly_balance["totalAssets"] - quarterly_balance["totalCurrentAssets"]
)
quarterly_balance["totalNonCurrentLiabilities"] = quarterly_balance["totalNonCurrentLiabilities"].fillna(
    quarterly_balance["totalLiabilities"] - quarterly_balance["totalCurrentLiabilities"]
)
quarterly_balance["netDebt"] = quarterly_balance["netDebt"].fillna(
    quarterly_balance["totalDebt"] - quarterly_balance["cashAndCashEquivalents"].fillna(0)
)
quarterly_balance = quarterly_balance.sort_values("date").reset_index(drop=True)
quarterly_balance["quarterLabel"] = quarterly_balance["period"].astype("string").str.upper().str.extract(r"(Q[1-4])", expand=False)
quarterly_balance.loc[quarterly_balance["quarterLabel"].isna(), "quarterLabel"] = (
    "Q" + quarterly_balance.loc[quarterly_balance["quarterLabel"].isna(), "date"].dt.quarter.astype("Int64").astype(str)
)
quarterly_balance["quarterlyYear"] = quarterly_balance["date"].dt.year
quarterly_balance["workingCapital"] = quarterly_balance["totalCurrentAssets"] - quarterly_balance["totalCurrentLiabilities"]
quarterly_balance["cashAndShortTermInvestmentsPctAssets"] = quarterly_balance["cashAndShortTermInvestments"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["receivablesPctAssets"] = quarterly_balance["netReceivables"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["inventoryPctAssets"] = quarterly_balance["inventory"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["ppePctAssets"] = quarterly_balance["propertyPlantEquipmentNet"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["goodwillIntangiblePctAssets"] = quarterly_balance["goodwillAndIntangibleAssets"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["longTermInvestmentsPctAssets"] = quarterly_balance["longTermInvestments"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["currentAssetsPctAssets"] = quarterly_balance["totalCurrentAssets"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["nonCurrentAssetsPctAssets"] = quarterly_balance["totalNonCurrentAssets"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["currentLiabilitiesPctAssets"] = quarterly_balance["totalCurrentLiabilities"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["nonCurrentLiabilitiesPctAssets"] = quarterly_balance["totalNonCurrentLiabilities"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["totalDebtPctAssets"] = quarterly_balance["totalDebt"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["totalLiabilitiesPctAssets"] = quarterly_balance["totalLiabilities"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["totalEquityPctAssets"] = quarterly_balance["totalEquity"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["currentRatio"] = quarterly_balance["totalCurrentAssets"].div(quarterly_balance["totalCurrentLiabilities"].replace(0, pd.NA))
quarterly_balance["cashRatio"] = quarterly_balance["cashAndShortTermInvestments"].div(quarterly_balance["totalCurrentLiabilities"].replace(0, pd.NA))
quarterly_balance["workingCapitalPctAssets"] = quarterly_balance["workingCapital"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["debtToAssets"] = quarterly_balance["totalDebt"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["liabilityToAssets"] = quarterly_balance["totalLiabilities"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["equityRatio"] = quarterly_balance["totalEquity"].div(quarterly_balance["totalAssets"].replace(0, pd.NA))
quarterly_balance["debtToEquity"] = quarterly_balance["totalDebt"].div(quarterly_balance["totalEquity"].replace(0, pd.NA))
quarterly_balance["netDebtToEquity"] = quarterly_balance["netDebt"].div(quarterly_balance["totalEquity"].replace(0, pd.NA))
quarterly_balance["totalAssetsYoY"] = quarterly_balance["totalAssets"].pct_change(4)
quarterly_balance["totalLiabilitiesYoY"] = quarterly_balance["totalLiabilities"].pct_change(4)
quarterly_balance["totalEquityYoY"] = quarterly_balance["totalEquity"].pct_change(4)
quarterly_balance["cashYoY"] = quarterly_balance["cashAndShortTermInvestments"].pct_change(4)
quarterly_balance["totalDebtYoY"] = quarterly_balance["totalDebt"].pct_change(4)
quarterly_balance["workingCapitalYoY"] = quarterly_balance["workingCapital"].pct_change(4)
quarterly_balance["currentRatioChange"] = quarterly_balance["currentRatio"].diff(4)
quarterly_balance["debtToEquityChange"] = quarterly_balance["debtToEquity"].diff(4)
quarterly_balance["liabilityToAssetsChange"] = quarterly_balance["liabilityToAssets"].diff(4)
quarterly_balance["equityRatioChange"] = quarterly_balance["equityRatio"].diff(4)

In [ ]:
# 5. Build annual summary stats
from IPython.display import display

latest_annual = annual_balance.iloc[-1]
annual_summary = pd.DataFrame(
    [
        {"metric": "Latest annual total assets", "value": latest_annual["totalAssets"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual cash and short-term investments", "value": latest_annual["cashAndShortTermInvestments"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual total debt", "value": latest_annual["totalDebt"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual net debt", "value": latest_annual["netDebt"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual total equity", "value": latest_annual["totalEquity"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual current ratio", "value": latest_annual["currentRatio"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual cash ratio", "value": latest_annual["cashRatio"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual debt to equity", "value": latest_annual["debtToEquity"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual equity ratio", "value": latest_annual["equityRatio"], "style": "percent", "asOf": latest_annual["date"].date()},
    ]
)
annual_summary_display = annual_summary.copy()
annual_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(annual_summary_display["value"], annual_summary_display["style"])]
display(annual_summary_display.loc[:, ["metric", "value", "asOf"]])

In [ ]:
# 6. Build quarterly summary stats
from IPython.display import display

latest_quarter = quarterly_balance.iloc[-1]
quarterly_summary = pd.DataFrame(
    [
        {"metric": "Latest quarterly total assets", "value": latest_quarter["totalAssets"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current assets", "value": latest_quarter["totalCurrentAssets"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current liabilities", "value": latest_quarter["totalCurrentLiabilities"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly working capital", "value": latest_quarter["workingCapital"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly current ratio", "value": latest_quarter["currentRatio"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly cash ratio", "value": latest_quarter["cashRatio"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly total debt", "value": latest_quarter["totalDebt"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly net debt", "value": latest_quarter["netDebt"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly debt to equity", "value": latest_quarter["debtToEquity"], "style": "ratio", "asOf": latest_quarter["date"].date()},
    ]
)
quarterly_summary_display = quarterly_summary.copy()
quarterly_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(quarterly_summary_display["value"], quarterly_summary_display["style"])]
display(quarterly_summary_display.loc[:, ["metric", "value", "asOf"]])

## Trend and Composition Charts

These cells move from summary tables into visualization. They cover annual and quarterly balance sheet scale, liquidity, composition, and growth so you can separate long-term capital structure drift from shorter-term shifts in working capital and leverage.

In [ ]:
# 7. Plot annual balance sheet trends
from plotly.subplots import make_subplots

annual_balance_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.24, 0.23, 0.23],
    subplot_titles=(
        "Annual Assets, Liabilities, and Equity" ,
        "Annual Current Assets, Current Liabilities, and Working Capital" ,
        "Annual Asset Mix as a Percent of Total Assets" ,
        "Annual Balance Sheet Growth Rates" ,
    ),
)

for metric_name, label, color in [
    ("totalAssets", "Total assets", "#60a5fa"),
    ("totalLiabilities", "Total liabilities", "#f59e0b"),
    ("totalEquity", "Total equity", "#34d399"),
]:
    annual_balance_fig.add_trace(
        go.Scatter(
            x=annual_balance["date"],
            y=annual_balance[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
        ),
        row=1,
        col=1,
    )

annual_balance_fig.add_trace(
    go.Bar(
        x=annual_balance["date"],
        y=annual_balance["totalCurrentAssets"],
        name="Current assets" ,
        marker_color="#38bdf8" ,
        opacity=0.55,
    ),
    row=2,
    col=1,
)
annual_balance_fig.add_trace(
    go.Bar(
        x=annual_balance["date"],
        y=annual_balance["totalCurrentLiabilities"],
        name="Current liabilities" ,
        marker_color="#fb7185" ,
        opacity=0.55,
    ),
    row=2,
    col=1,
)
annual_balance_fig.add_trace(
    go.Scatter(
        x=annual_balance["date"],
        y=annual_balance["workingCapital"],
        name="Working capital" ,
        mode="lines+markers" ,
        line={"color": "#a78bfa", "width": 2.5},
        marker={"size": 7},
    ),
    row=2,
    col=1,
)

for metric_name, label, color in [
    ("cashAndShortTermInvestmentsPctAssets", "Cash and STI % assets", "#60a5fa"),
    ("receivablesPctAssets", "Receivables % assets", "#22c55e"),
    ("inventoryPctAssets", "Inventory % assets", "#f59e0b"),
    ("ppePctAssets", "Net PP&E % assets", "#f97316"),
    ("longTermInvestmentsPctAssets", "Long-term investments % assets", "#a78bfa"),
    ("goodwillIntangiblePctAssets", "Goodwill and intangibles % assets", "#fb7185"),
]:
    if annual_balance[metric_name].notna().any():
        annual_balance_fig.add_trace(
            go.Scatter(
                x=annual_balance["date"],
                y=annual_balance[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.25},
                marker={"size": 6},
            ),
            row=3,
            col=1,
        )

for metric_name, label, color in [
    ("totalAssetsYoY", "Assets growth", "#60a5fa"),
    ("totalLiabilitiesYoY", "Liabilities growth", "#f59e0b"),
    ("totalEquityYoY", "Equity growth", "#34d399"),
    ("totalDebtYoY", "Debt growth", "#fb7185"),
]:
    annual_balance_fig.add_trace(
        go.Scatter(
            x=annual_balance["date"],
            y=annual_balance[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
        ),
        row=4,
        col=1,
    )

apply_standard_figure_layout(annual_balance_fig, f"{chart_label} annual balance sheet", 1420)
annual_balance_fig.update_layout(barmode="group")

for row_number in [1, 2, 3, 4]:
    annual_balance_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=1)

annual_balance_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
annual_balance_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=2, col=1)
annual_balance_fig.update_yaxes(title_text="Percent of assets", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=3, col=1)
annual_balance_fig.update_yaxes(title_text="YoY growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
annual_balance_fig.update_xaxes(title_text="Date", row=4, col=1)

annual_balance_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly balance sheet trends
from plotly.subplots import make_subplots

quarterly_balance_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.24, 0.23, 0.23],
    subplot_titles=(
        "Quarterly Assets, Liabilities, and Equity" ,
        "Quarterly Current Assets, Current Liabilities, and Working Capital" ,
        "Quarterly Asset Mix as a Percent of Total Assets" ,
        "Quarterly Balance Sheet Growth Rates" ,
    ),
)

for metric_name, label, color in [
    ("totalAssets", "Total assets", "#60a5fa"),
    ("totalLiabilities", "Total liabilities", "#f59e0b"),
    ("totalEquity", "Total equity", "#34d399"),
]:
    quarterly_balance_fig.add_trace(
        go.Scatter(
            x=quarterly_balance["date"],
            y=quarterly_balance[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.25},
            marker={"size": 6},
        ),
        row=1,
        col=1,
    )

quarterly_balance_fig.add_trace(
    go.Bar(
        x=quarterly_balance["date"],
        y=quarterly_balance["totalCurrentAssets"],
        name="Current assets" ,
        marker_color="#38bdf8" ,
        opacity=0.5,
    ),
    row=2,
    col=1,
)
quarterly_balance_fig.add_trace(
    go.Bar(
        x=quarterly_balance["date"],
        y=quarterly_balance["totalCurrentLiabilities"],
        name="Current liabilities" ,
        marker_color="#fb7185" ,
        opacity=0.5,
    ),
    row=2,
    col=1,
)
quarterly_balance_fig.add_trace(
    go.Scatter(
        x=quarterly_balance["date"],
        y=quarterly_balance["workingCapital"],
        name="Working capital" ,
        mode="lines+markers" ,
        line={"color": "#a78bfa", "width": 2.25},
        marker={"size": 6},
    ),
    row=2,
    col=1,
)

for metric_name, label, color in [
    ("cashAndShortTermInvestmentsPctAssets", "Cash and STI % assets", "#60a5fa"),
    ("receivablesPctAssets", "Receivables % assets", "#22c55e"),
    ("inventoryPctAssets", "Inventory % assets", "#f59e0b"),
    ("ppePctAssets", "Net PP&E % assets", "#f97316"),
    ("longTermInvestmentsPctAssets", "Long-term investments % assets", "#a78bfa"),
    ("goodwillIntangiblePctAssets", "Goodwill and intangibles % assets", "#fb7185"),
]:
    if quarterly_balance[metric_name].notna().any():
        quarterly_balance_fig.add_trace(
            go.Scatter(
                x=quarterly_balance["date"],
                y=quarterly_balance[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.0},
                marker={"size": 6},
            ),
            row=3,
            col=1,
        )

for metric_name, label, color in [
    ("totalAssetsYoY", "Assets growth", "#60a5fa"),
    ("totalLiabilitiesYoY", "Liabilities growth", "#f59e0b"),
    ("totalEquityYoY", "Equity growth", "#34d399"),
    ("totalDebtYoY", "Debt growth", "#fb7185"),
]:
    quarterly_balance_fig.add_trace(
        go.Scatter(
            x=quarterly_balance["date"],
            y=quarterly_balance[metric_name],
            name=label,
            mode="lines+markers" ,
            line={"color": color, "width": 2.0},
            marker={"size": 6},
        ),
        row=4,
        col=1,
    )

apply_standard_figure_layout(quarterly_balance_fig, f"{chart_label} quarterly balance sheet", 1420)
quarterly_balance_fig.update_layout(barmode="group")

for row_number in [1, 2, 3, 4]:
    quarterly_balance_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=1)

quarterly_balance_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
quarterly_balance_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat="$,.3s", row=2, col=1)
quarterly_balance_fig.update_yaxes(title_text="Percent of assets", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=3, col=1)
quarterly_balance_fig.update_yaxes(title_text="YoY growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
quarterly_balance_fig.update_xaxes(title_text="Date", row=4, col=1)

quarterly_balance_fig.show(config={"responsive": True, "displaylogo": False})

## Common-Size, Liquidity, and Capital Structure

These cells recast the balance sheet as a share of total assets, then separate liquidity from leverage so you can see whether changes are being driven by asset composition, working-capital pressure, or capital-structure decisions.

In [ ]:
# 9. Build common-size balance sheet views
from IPython.display import display

def build_common_size_balance_sheet(frame: pd.DataFrame, column_labels: pd.Series) -> pd.DataFrame:
    total_assets = frame["totalAssets"].replace(0, pd.NA)
    statement_rows = [
        ("Cash and short-term investments", frame["cashAndShortTermInvestments"].div(total_assets)),
        ("Receivables", frame["netReceivables"].div(total_assets)),
        ("Inventory", frame["inventory"].div(total_assets)),
        ("Total current assets", frame["currentAssetsPctAssets"]),
        ("Net PP&E", frame["propertyPlantEquipmentNet"].div(total_assets)),
        ("Long-term investments", frame["longTermInvestments"].div(total_assets)),
        ("Goodwill and intangibles", frame["goodwillAndIntangibleAssets"].div(total_assets)),
        ("Total non-current assets", frame["nonCurrentAssetsPctAssets"]),
        ("Total assets", pd.Series(1.0, index=frame.index)),
        ("Accounts payable", frame["accountPayables"].div(total_assets)),
        ("Short-term debt", frame["shortTermDebt"].div(total_assets)),
        ("Total current liabilities", frame["currentLiabilitiesPctAssets"]),
        ("Long-term debt", frame["longTermDebt"].div(total_assets)),
        ("Total debt", frame["totalDebtPctAssets"]),
        ("Total non-current liabilities", frame["nonCurrentLiabilitiesPctAssets"]),
        ("Total liabilities", frame["totalLiabilitiesPctAssets"]),
        ("Total equity", frame["totalEquityPctAssets"]),
    ]

    filtered_rows = []
    for label, values in statement_rows:
        if label in {"Total assets", "Total liabilities", "Total equity"} or values.fillna(0).abs().gt(1e-6).any():
            filtered_rows.append((label, values))

    statement = pd.DataFrame(
        [values.reset_index(drop=True).tolist() for _, values in filtered_rows],
        index=[label for label, _ in filtered_rows],
        columns=column_labels.reset_index(drop=True).tolist(),
    )
    return statement.dropna(how="all", axis=0)

annual_common_size_frame = annual_balance.tail(min(len(annual_balance), 6)).copy()
annual_common_size_labels = annual_common_size_frame["calendarYear"].astype("Int64").astype(str)
annual_common_size = build_common_size_balance_sheet(annual_common_size_frame, annual_common_size_labels)

quarterly_common_size_frame = quarterly_balance.tail(min(len(quarterly_balance), 8)).copy()
quarterly_common_size_labels = (
    quarterly_common_size_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_common_size_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_common_size_frame["date"].dt.strftime("%b %Y")
    + ")"
)
quarterly_common_size = build_common_size_balance_sheet(quarterly_common_size_frame, quarterly_common_size_labels)

display(annual_common_size.style.format("{:.1%}").set_caption(f"{SYMBOL} annual common-size balance sheet (% of total assets)"))
display(quarterly_common_size.style.format("{:.1%}").set_caption(f"{SYMBOL} quarterly common-size balance sheet (% of total assets)"))

In [ ]:
# 10. Plot liquidity and capital structure
from plotly.subplots import make_subplots

balance_sheet_risk_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    subplot_titles=(
        "Annual cash versus debt" ,
        "Quarterly cash versus debt" ,
        "Annual liquidity ratios" ,
        "Quarterly liquidity ratios" ,
        "Annual capital structure percentages" ,
        "Quarterly capital structure percentages" ,
    ),
)

for frame, col_number in [(annual_balance, 1), (quarterly_balance, 2)]:
    for metric_name, label, color in [
        ("cashAndShortTermInvestments", "Cash and STI", "#60a5fa"),
        ("totalDebt", "Total debt", "#f59e0b"),
        ("netDebt", "Net debt", "#f43f5e"),
    ]:
        balance_sheet_risk_fig.add_trace(
            go.Scatter(
                x=frame["date"],
                y=frame[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.5},
                marker={"size": 7 if col_number == 1 else 6},
                legendgroup=metric_name,
                showlegend=col_number == 1,
            ),
            row=1,
            col=col_number,
        )

    for metric_name, label, color in [
        ("currentRatio", "Current ratio", "#22c55e"),
        ("cashRatio", "Cash ratio", "#a78bfa"),
    ]:
        balance_sheet_risk_fig.add_trace(
            go.Scatter(
                x=frame["date"],
                y=frame[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.25},
                marker={"size": 7 if col_number == 1 else 6},
                legendgroup=metric_name,
                showlegend=False,
            ),
            row=2,
            col=col_number,
        )

    for metric_name, label, color in [
        ("debtToAssets", "Debt % assets", "#f59e0b"),
        ("liabilityToAssets", "Liabilities % assets", "#fb7185"),
        ("equityRatio", "Equity % assets", "#34d399"),
        ("workingCapitalPctAssets", "Working capital % assets", "#38bdf8"),
    ]:
        balance_sheet_risk_fig.add_trace(
            go.Scatter(
                x=frame["date"],
                y=frame[metric_name],
                name=label,
                mode="lines+markers" ,
                line={"color": color, "width": 2.25},
                marker={"size": 7 if col_number == 1 else 6},
                legendgroup=metric_name,
                showlegend=False,
            ),
            row=3,
            col=col_number,
        )

apply_standard_figure_layout(balance_sheet_risk_fig, f"{chart_label} liquidity and capital structure", 1280)

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        balance_sheet_risk_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=row_number, col=col_number)

balance_sheet_risk_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
balance_sheet_risk_fig.update_yaxes(title_text="Amount (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=2)
balance_sheet_risk_fig.update_yaxes(title_text="Ratio", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=",.2f", row=2, col=1)
balance_sheet_risk_fig.update_yaxes(title_text="Ratio", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=",.2f", row=2, col=2)
balance_sheet_risk_fig.update_yaxes(title_text="Percent of assets", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=3, col=1)
balance_sheet_risk_fig.update_yaxes(title_text="Percent of assets", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=3, col=2)

balance_sheet_risk_fig.show(config={"responsive": True, "displaylogo": False})